# Setup and first runs (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smhasibul0/dse-anomaly-detection/blob/main/notebooks/00_colab_setup.ipynb)

Run top to bottom. What it does:
1. Clones (or updates) the repo and installs `requirements.txt`.
2. Mounts Google Drive and keeps all data in `MyDrive/<DRIVE_FOLDER>/data/`, so downloads survive between Colab sessions.
3. Fixes the seed from `config.yaml`.
4. Pulls DSEX and the default DSE stocks, then runs the quality checks and the go/no-go test.
5. Runs the first GARCH workflow on S&P 500 (`^GSPC`, yfinance).

Code lives in git. Data lives in Drive and is never committed.

## 1. Settings
Defaults point at the team repo; change only if you fork it. For a **private** repo, add a GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar, fine-grained token with read access to this repo).

In [ ]:
REPO_URL = "https://github.com/smhasibul0/dse-anomaly-detection.git"
BRANCH = "main"
DRIVE_FOLDER = "dse-anomaly"                         # folder inside MyDrive shared by the team

## 2. Clone or update the repo

In [ ]:
import os, subprocess
from pathlib import Path

REPO_DIR = Path("/content") / Path(REPO_URL).stem
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # public repo, or running outside Colab

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Working in", os.getcwd())
!git log --oneline -3

## 3. Install requirements

In [ ]:
!pip install -q -r requirements.txt

## 4. Mount Google Drive for data
Everything under `data/` is read from and written to Drive. Scripts pick this up through the `PROJECT_DATA_DIR` environment variable (see `src/paths.py`).

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive") / DRIVE_FOLDER / "data"
except ImportError:
    DATA_DIR = REPO_DIR / "data"  # local run
for sub in ["raw", "interim", "processed"]:
    (DATA_DIR / sub).mkdir(parents=True, exist_ok=True)
os.environ["PROJECT_DATA_DIR"] = str(DATA_DIR)
print("Data dir:", DATA_DIR)

## 5. Fix the seed
Every script also calls `set_seed` itself; this cell seeds the notebook process for anything run interactively. See `docs/seed_policy.md`.

In [ ]:
import sys, yaml
sys.path.insert(0, str(REPO_DIR))
from src.seeds import set_seed

cfg = yaml.safe_load(open("config.yaml"))
SEED = cfg["seed"]
set_seed(SEED)
print("Seed:", SEED)

## 6. Pull DSE data
DSEX (from its 2013 launch) plus the default stock list in `scripts/fetch_dse.py`. Prices are **unadjusted**. Each symbol is fetched in yearly chunks with a 1 s pause, so this takes several minutes. It is skipped if `dsex.csv` is already in Drive; set `FORCE_FETCH = True` to pull again.

In [ ]:
FORCE_FETCH = False
DSE_RAW = DATA_DIR / "raw" / "dse"
if FORCE_FETCH or not (DSE_RAW / "dsex.csv").exists():
    !python scripts/fetch_dse.py --start 2013-01-01 --out "{DSE_RAW}"
else:
    print("Already pulled:", sorted(p.name for p in DSE_RAW.glob("*.csv")))

## 7. DSE quality checks
Writes `qc_summary.csv`, `adj_events.csv` (likely bonus/split days) and back-adjusted prices to `data/interim/dse_qc/`.

In [ ]:
DSE_QC = DATA_DIR / "interim" / "dse_qc"
!python scripts/qc_dse.py --raw "{DSE_RAW}" --out "{DSE_QC}"
import pandas as pd
qc = pd.read_csv(DSE_QC / "qc_summary.csv")
qc

### Go/no-go
From `docs/dse_data_sources.md`: the daily DSE plan stands if DSEX has at least 3,000 rows from 2013 **and** at least 10 stocks have 10+ years with under 2% missing days. The missing-day share below counts floor-price periods too, so it is slightly stricter than the rule.

In [ ]:
dsex_rows = int(qc.loc[qc.series == "DSEX", "rows"].sum())
st = qc[qc.series != "DSEX"].copy()
st["missing_pct"] = 100 * st["missing_vs_calendar"] / (st["rows"] + st["missing_vs_calendar"])
ok = st[(st["years"] >= 10) & (st["missing_pct"] < 2)]
print(f"DSEX rows: {dsex_rows}  (need >= 3000)")
print(f"Stocks with 10+ years and <2% missing: {len(ok)}  (need >= 10)")
print(ok[["series", "years", "missing_pct", "median_daily_value_mn"]].sort_values("median_daily_value_mn", ascending=False).to_string(index=False))
print("\nVERDICT:", "GO" if dsex_rows >= 3000 and len(ok) >= 10 else "NO-GO: consider the India (NSE) fallback")

## 8. First GARCH run on S&P 500
Downloads `^GSPC` 2010 to 2025 via yfinance, fits GARCH / EGARCH / GJR on the training split only, checks residuals and flags extreme standardized residuals. Settings are in `config.yaml`.

In [ ]:
!python scripts/01_first_garch.py

In [ ]:
from IPython.display import Image, display
display(Image("reports/first_garch/first_garch.png"))
pd.read_csv("reports/first_garch/residual_diagnostics.csv")

## 9. Saving work
- Data is already in Drive.
- Reports are in the repo's `reports/` folder. To keep them, commit from your laptop, or copy them to Drive:

In [ ]:
!mkdir -p "{DATA_DIR.parent}/reports" && cp -r reports/* "{DATA_DIR.parent}/reports/" && echo "copied to {DATA_DIR.parent}/reports" 